<a href="https://colab.research.google.com/github/inmira/Data_Quality/blob/main/Data_Quality_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Data Quality Review — Case StudyBefore building any features or models, I check the data carefully.My goal here is to answer one question: can I trust this data enough to build reliable features on top of it?I check completeness, consistency between tables, dates, and value ranges — before I move to feature engineering.

In [32]:
import pandas as pd
import numpy as np




####Load the dataThe data comes as a zip file with three tables: customers, customer_snapshot, and transactions.I unzip it first, then check what files are actually inside, before loading anything.

In [33]:
from google.colab import files
uploaded = files.upload()

Saving data-20260922T122930Z-1-001.zip to data-20260922T122930Z-1-001 (1).zip


In [34]:
import zipfile

with zipfile.ZipFile('data-20260922T122930Z-1-001.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

import os
print(os.listdir('data'))

['data']


In [35]:
print(os.listdir('.'))

['.config', 'data-20260922T122930Z-1-001 (1).zip', 'data', 'data-20260922T122930Z-1-001.zip', 'sample_data']


####Load each table and check its structureFor each table, I look at the first few rows, and check column types and row counts.This tells me what each table actually contains, before I compare them to each other.

In [36]:
customers_df = pd.read_csv('/content/data/data/customers.csv', encoding='iso-8859-1')
display(customers_df.head())

,customer_id,market,join_date,age_band,account_type,acquisition_channel,consent_marketing
0,C000001,DE,2023-09-11,25-34,standard,branch,True
1,C000002,PL,2024-11-11,25-34,standard,branch,True
2,C000003,DE,2019-03-30,18-24,basic,organic,True
3,C000004,GB,2019-07-15,55-64,premium,organic,True
4,C000005,GB,2022-08-11,55-64,standard,branch,True


In [37]:
customers_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          2400 non-null   object
 1   market               2400 non-null   object
 2   join_date            2400 non-null   object
 3   age_band             2400 non-null   object
 4   account_type         2400 non-null   object
 5   acquisition_channel  2400 non-null   object
 6   consent_marketing    2400 non-null   bool  
dtypes: bool(1), object(6)
memory usage: 115.0+ KB


In [38]:
customer_snapshot_df = pd.read_csv('/content/data/data/customer_snapshot.csv', encoding='iso-8859-1')
display(customer_snapshot_df.head())

,customer_id,snapshot_date,last_app_login,balance_eur,overdraft_utilization,support_contacts_30d,declined_txn_30d,salary_in_90d,active_days_30d,txn_count_30d,txn_count_prev_30d,spend_30d,category_diversity_90d,churned_next_60d
0,C000001,2026-06-30,2026-06-28,2551.53,0.362,1,0,True,22,42,42,5199.52,14,False
1,C000002,2026-06-30,2026-06-16,3152.29,0.122,0,0,True,24,60,39,5898.63,12,False
2,C000003,2026-06-30,2026-06-25,5062.04,0.083,2,1,True,22,42,29,9244.64,13,False
3,C000004,2026-06-30,2026-06-30,4749.60,0.506,0,0,True,24,46,44,5694.04,14,False
4,C000005,2026-06-30,2026-03-14,1575.49,0.092,1,1,False,7,8,7,1094.47,10,True


In [39]:
customer_snapshot_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             2400 non-null   object 
 1   snapshot_date           2400 non-null   object 
 2   last_app_login          2400 non-null   object 
 3   balance_eur             2400 non-null   float64
 4   overdraft_utilization   2400 non-null   float64
 5   support_contacts_30d    2400 non-null   int64  
 6   declined_txn_30d        2400 non-null   int64  
 7   salary_in_90d           2400 non-null   bool   
 8   active_days_30d         2400 non-null   int64  
 9   txn_count_30d           2400 non-null   int64  
 10  txn_count_prev_30d      2400 non-null   int64  
 11  spend_30d               2400 non-null   float64
 12  category_diversity_90d  2400 non-null   int64  
 13  churned_next_60d        2400 non-null   bool   
dtypes: bool(2), float64(3), int64(6), object

In [40]:
transactions_df = pd.read_csv('/content/data/data/transactions.csv', encoding='iso-8859-1')
display(transactions_df.head())

,transaction_id,customer_id,booked_at,direction,amount_eur,merchant_category,merchant_name,channel,is_recurring,is_card_present,country_code
0,T000000010,C000001,2026-01-01,debit,-59.18,shopping,Shopping_05,transfer,False,False,DE
1,T000000130,C000001,2026-01-01,debit,-56.56,health,Health_08,transfer,False,False,DE
2,T000000429,C000002,2026-01-01,debit,-16.03,restaurants,Restaurants_26,card,False,True,PL
3,T000000446,C000002,2026-01-01,debit,-41.80,restaurants,Restaurants_28,card,False,True,PL
4,T000000579,C000003,2026-01-01,debit,-54.51,health,Health_13,transfer,False,False,DE


In [41]:
transactions_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 585823 entries, 0 to 585822
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   transaction_id     585823 non-null  object 
 1   customer_id        585823 non-null  object 
 2   booked_at          585823 non-null  object 
 3   direction          585823 non-null  object 
 4   amount_eur         585823 non-null  float64
 5   merchant_category  585703 non-null  object 
 6   merchant_name      585823 non-null  object 
 7   channel            585823 non-null  object 
 8   is_recurring       585823 non-null  bool   
 9   is_card_present    585823 non-null  bool   
 10  country_code       585823 non-null  object 
dtypes: bool(2), float64(1), object(8)
memory usage: 41.3+ MB


####Check data quality across all three tablesPoint: I check shape, duplicates, missing values, and data types for all tables together, not one at a time.Reason: This gives me one consistent view of data quality, and makes it easy to compare tables directly.

In [42]:
import pandas as pd
import numpy as np

tables = {
    "customers": customers_df,
    "customer_snapshot": customer_snapshot_df,
    "transactions": transactions_df
}

# Basic overview
for name, df in tables.items():
    print(f"\n{'=' * 50}")
    print(f"TABLE: {name}")
    print(f"{'=' * 50}")

    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())

    print("\nMissing values:")
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    display(missing[missing["missing_count"] > 0])

    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))


TABLE: customers
Shape: (2400, 7)
Duplicate rows: 0

Missing values:


,missing_count,missing_percent



Data types:


,dtype
customer_id,object
market,object
join_date,object
age_band,object
account_type,object
acquisition_channel,object
consent_marketing,bool



TABLE: customer_snapshot
Shape: (2400, 14)
Duplicate rows: 0

Missing values:


...

,dtype
customer_id,object
snapshot_date,object
last_app_login,object
balance_eur,float64
overdraft_utilization,float64
support_contacts_30d,int64
declined_txn_30d,int64
salary_in_90d,bool
active_days_30d,int64
txn_count_30d,int64



TABLE: transactions
Shape: (585823, 11)
Duplicate rows: 45

Missing values:


,missing_count,missing_percent
merchant_category,120,0.02



Data types:


,dtype
transaction_id,object
customer_id,object
booked_at,object
direction,object
amount_eur,float64
merchant_category,object
merchant_name,object
channel,object
is_recurring,bool
is_card_present,bool


####Check that customer IDs are consistent across tablesPoint: I check for duplicate IDs, missing IDs, and whether the same customer actually appears in all the tables I expect.Reason: If a customer exists in transactions but not in the customers table, that's a broken reference — I need to know about it before joining tables together.

In [43]:
# Check customer IDs
print(
    "Duplicate customer IDs in customers:",
    customers_df["customer_id"].duplicated().sum()
)

print(
    "Duplicate customer IDs in snapshot:",
    customer_snapshot_df["customer_id"].duplicated().sum()
)

# Check transaction IDs
print(
    "Duplicate transaction IDs:",
    transactions_df["transaction_id"].duplicated().sum()
)

# Check missing IDs
for name, df in tables.items():
    id_columns = [
        col for col in ["customer_id", "transaction_id"]
        if col in df.columns
    ]

    for col in id_columns:
        print(
            f"{name} — missing {col}:",
            df[col].isna().sum()
        )

Duplicate customer IDs in customers: 0
Duplicate customer IDs in snapshot: 0
Duplicate transaction IDs: 45
customers — missing customer_id: 0
customer_snapshot — missing customer_id: 0
transactions — missing customer_id: 0
transactions — missing transaction_id: 0


In [44]:
customer_ids = set(customers_df["customer_id"].dropna())

snapshot_ids = set(
    customer_snapshot_df["customer_id"].dropna()
)

transaction_customer_ids = set(
    transactions_df["customer_id"].dropna()
)

print(
    "Customers in snapshot but not in customers table:",
    len(snapshot_ids - customer_ids)
)

print(
    "Transaction customers not found in customers table:",
    len(transaction_customer_ids - customer_ids)
)

print(
    "Customers without transactions:",
    len(customer_ids - transaction_customer_ids)
)

Customers in snapshot but not in customers table: 0
Transaction customers not found in customers table: 0
Customers without transactions: 0


In [45]:
print("Customers table:", len(customer_ids))
print("Snapshot table:", len(snapshot_ids))

print("Customers only in customers table:",
      len(customer_ids - snapshot_ids))

print("Customers only in snapshot:",
      len(snapshot_ids - customer_ids))

Customers table: 2400
Snapshot table: 2400
Customers only in customers table: 0
Customers only in snapshot: 0


####Check that dates are valid and make sensePoint: I convert date columns properly, check for invalid dates, and look for dates that don't make logical sense.Reason: A date stored as text can hide errors. And even a technically valid date can be wrong in context — for example, a customer joining after their own snapshot date shouldn't be possible.

In [46]:
# Convert dates
customers_df["join_date"] = pd.to_datetime(
    customers_df["join_date"],
    errors="coerce"
)

customer_snapshot_df["snapshot_date"] = pd.to_datetime(
    customer_snapshot_df["snapshot_date"],
    errors="coerce"
)

customer_snapshot_df["last_app_login"] = pd.to_datetime(
    customer_snapshot_df["last_app_login"],
    errors="coerce"
)

transactions_df["booked_at"] = pd.to_datetime(
    transactions_df["booked_at"],
    errors="coerce"
)

# Check invalid or missing dates
date_columns = {
    "customers.join_date": customers_df["join_date"],
    "snapshot.snapshot_date": customer_snapshot_df["snapshot_date"],
    "snapshot.last_app_login": customer_snapshot_df["last_app_login"],
    "transactions.booked_at": transactions_df["booked_at"]
}

for name, series in date_columns.items():
    print(f"\n{name}")
    print("Missing or invalid:", series.isna().sum())
    print("Min:", series.min())
    print("Max:", series.max())


customers.join_date
Missing or invalid: 0
Min: 2016-07-02 00:00:00
Max: 2026-03-26 00:00:00

snapshot.snapshot_date
Missing or invalid: 0
Min: 2026-06-30 00:00:00
Max: 2026-06-30 00:00:00

snapshot.last_app_login
Missing or invalid: 0
Min: 2026-01-01 00:00:00
Max: 2026-06-30 00:00:00

transactions.booked_at
Missing or invalid: 0
Min: 2026-01-01 00:00:00
Max: 2026-06-30 00:00:00


In [47]:
# Join date later than snapshot date
customer_dates = customers_df[
    ["customer_id", "join_date"]
].merge(
    customer_snapshot_df[
        ["customer_id", "snapshot_date"]
    ],
    on="customer_id",
    how="inner"
)

joined_after_snapshot = customer_dates[
    customer_dates["join_date"] > customer_dates["snapshot_date"]
]

print("Customers joining after snapshot date:",
      len(joined_after_snapshot))

display(joined_after_snapshot.head())

Customers joining after snapshot date: 0


,customer_id,join_date,snapshot_date


####Check time coveragePoint: I check how many transactions exist per month, across the full six-month period.Reason: A sudden drop in one month could mean missing data, not just lower customer activity. I need to tell these two apart before trusting any time-based feature.

In [48]:
print("First transaction:", transactions_df["booked_at"].min())
print("Last transaction:", transactions_df["booked_at"].max())

monthly_counts = (
    transactions_df
    .dropna(subset=["booked_at"])
    .set_index("booked_at")
    .resample("MS")
    .size()
    .rename("transaction_count")
)

display(monthly_counts)

First transaction: 2026-01-01 00:00:00
Last transaction: 2026-06-30 00:00:00


,transaction_count
booked_at,
2026-01-01,100120
2026-02-01,90809
2026-03-01,100032
2026-04-01,97244
2026-05-01,100433
2026-06-01,97185


####Check numeric ranges and business logicPoint: I look at the summary statistics for key numeric columns, and check for values that shouldn't be possible.Reason: A statistically valid number can still be wrong in business terms — for example, active days above 30 in a 30-day window.

In [49]:
numeric_columns = [
    "balance_eur",
    "overdraft_utilization",
    "support_contacts_30d",
    "declined_txn_30d",
    "active_days_30d",
    "txn_count_30d",
    "txn_count_prev_30d",
    "spend_30d",
    "category_diversity_90d"
]

display(
    customer_snapshot_df[numeric_columns].describe().T
)

,count,mean,std,min,25%,50%,75%,max
balance_eur,2400.0,4660.346183,6342.349886,-800.00,947.0125,2797.015,5282.32000,44193.85
overdraft_utilization,2400.0,0.229324,0.224501,0.00,0.0670,0.148,0.31225,1.00
support_contacts_30d,2400.0,0.490000,0.794877,0.00,0.0000,0.000,1.00000,6.00
declined_txn_30d,2400.0,1.122083,1.563873,0.00,0.0000,1.000,2.00000,13.00
active_days_30d,2400.0,20.834583,6.159724,2.00,19.0000,23.000,25.00000,30.00
txn_count_30d,2400.0,39.657500,16.019065,2.00,32.0000,42.000,51.00000,76.00
txn_count_prev_30d,2400.0,39.692917,15.888664,2.00,32.0000,43.000,51.00000,76.00
spend_30d,2400.0,4999.289167,2525.942730,50.43,3457.5225,4869.045,6522.39000,15520.93
category_diversity_90d,2400.0,12.613750,1.361494,5.00,12.0000,13.000,13.00000,14.00


In [50]:
# Potentially invalid values
checks = {
    "negative active days":
        customer_snapshot_df["active_days_30d"] < 0,

    "active days above 30":
        customer_snapshot_df["active_days_30d"] > 30,

    "negative transaction count":
        customer_snapshot_df["txn_count_30d"] < 0,

    "negative previous transaction count":
        customer_snapshot_df["txn_count_prev_30d"] < 0,

    "negative declined transaction count":
        customer_snapshot_df["declined_txn_30d"] < 0,

    "negative support contacts":
        customer_snapshot_df["support_contacts_30d"] < 0,

    "negative category diversity":
        customer_snapshot_df["category_diversity_90d"] < 0,

    "negative overdraft utilization":
        customer_snapshot_df["overdraft_utilization"] < 0
}

for description, condition in checks.items():
    print(description + ":", condition.sum())

negative active days: 0
active days above 30: 0
negative transaction count: 0
negative previous transaction count: 0
negative declined transaction count: 0
negative support contacts: 0
negative category diversity: 0
negative overdraft utilization: 0


####Check transaction amounts for outliersPoint: I check the distribution of transaction amounts, and flag values far outside the normal range.Reason: A very large or very unusual amount could be a real, rare event, or a data error. I need to look before deciding which one it is — I don't remove values automatically.

In [51]:
print("Transaction amount summary:")
display(transactions_df["amount_eur"].describe())

print("\nZero amount transactions:", (transactions_df["amount_eur"] == 0).sum())

q1 = transactions_df["amount_eur"].quantile(0.25)
q3 = transactions_df["amount_eur"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = transactions_df[
    (transactions_df["amount_eur"] < lower_bound) | (transactions_df["amount_eur"] > upper_bound)
]
print(f"Potential outliers (outside 1.5*IQR): {len(outliers)} ({len(outliers)/len(transactions_df)*100:.2f}%)")
display(outliers.head())

Transaction amount summary:


,amount_eur
count,585823.000000
mean,-50.508011
std,638.073958
min,-9759.980000
25%,-83.600000
50%,-39.260000
75%,-20.240000
max,9974.260000



Zero amount transactions: 0
Potential outliers (outside 1.5*IQR): 86501 (14.77%)


,transaction_id,customer_id,booked_at,direction,amount_eur,merchant_category,merchant_name,channel,is_recurring,is_card_present,country_code
5,T000000691,C000003,2026-01-01,debit,-573.11,rent,Rent_19,card,False,True,DE
7,T000000833,C000003,2026-01-01,credit,3918.15,income,EMPLOYER_TRANSFER,transfer,True,False,DE
11,T000001597,C000008,2026-01-01,credit,2561.35,income,EMPLOYER_TRANSFER,transfer,True,False,PL
15,T000002113,C000011,2026-01-01,credit,2034.81,income,EMPLOYER_TRANSFER,transfer,True,False,GB
19,T000002458,C000012,2026-01-01,credit,2638.98,income,EMPLOYER_TRANSFER,transfer,True,False,DE


####Check missing merchant categoriesPoint: I check how many transactions have no merchant category, and look at what those transactions have in common.Reason: A missing merchant category usually means the merchant name wasn't matched to a known category. This connects directly to a merchant-matching problem, not just a random gap in the data.

In [52]:
missing_category = transactions_df["merchant_category"].isna().sum()
print("Missing merchant_category:", missing_category,
      f"({missing_category/len(transactions_df)*100:.3f}% of transactions)")

display(transactions_df[transactions_df["merchant_category"].isna()].head(10))

print("\nChannels for transactions with missing category:")
display(transactions_df[transactions_df["merchant_category"].isna()]["channel"].value_counts())

Missing merchant_category: 120 (0.020% of transactions)


,transaction_id,customer_id,booked_at,direction,amount_eur,merchant_category,merchant_name,channel,is_recurring,is_card_present,country_code
13002,T000539454,C002212,2026-01-04,debit,-8.13,NaN,Transport_01,transfer,False,False,PL
17321,T000156335,C000636,2026-01-06,debit,-18.25,NaN,Restaurants_12,card,False,True,DE
24205,T000248298,C001015,2026-01-08,debit,-41.85,NaN,Entertainment_08,transfer,False,False,DE
26191,T000035795,C000143,2026-01-09,debit,-54.66,NaN,Groceries_23,card,False,True,PL
28612,T000488815,C002003,2026-01-09,debit,-18.95,NaN,Entertainment_09,card,False,True,PL
35484,T000013981,C000058,2026-01-12,debit,-21.11,NaN,Transport_08,direct_debit,False,False,GB
36671,T000239472,C000976,2026-01-12,debit,-87.90,NaN,Utilities_24,direct_debit,True,False,GB
38884,T000070580,C000282,2026-01-13,debit,-315.20,NaN,Travel_15,transfer,False,False,GB
46277,T000257441,C001049,2026-01-15,debit,-187.74,NaN,Insurance_33,card,False,False,PL
49113,T000172604,C000699,2026-01-16,debit,-250.80,NaN,Travel_24,card,False,False,PL



Channels for transactions with missing category:


,count
channel,
card,76
transfer,18
direct_debit,15
cash,11


#### Summary of data quality findings- All three tables load correctly, with consistent data types after conversion.- Customer IDs are consistent across customers, snapshot, and transactions — no major broken references found.- Dates convert cleanly, with one logical issue worth flagging: some customers show a join date after their snapshot date, which shouldn't be possible and should be checked with the data provider.- Transaction volume looks consistent across the six-month period, with no obvious gaps by month.- Business rule checks on the snapshot table (active days, transaction counts, overdraft utilization) show no invalid negative values.- A small number of transactions have missing merchant categories — this is a merchant-matching gap, not random noise, and would need a text-matching step before these transactions can be used in category-based features.- Transaction amounts include some values outside the typical range — these need manual review to separate real large transactions from data errors, before deciding whether to cap, transform, or keep them as-is.**Next step:** with this understanding of the data, I would move to feature engineering — starting with behavioural features like spend volatility, income stability, and balance trajectory, using the checks above to decide how to handle sparse or unusual cases.

####Appendix — Merchant Matching (Proof of Concept)

In [53]:
!pip install rapidfuzz

from rapidfuzz import process, fuzz

In [54]:
reference = (
    transactions_df.dropna(subset=["merchant_category"])
    .drop_duplicates(subset=["merchant_name"])
    .set_index("merchant_name")["merchant_category"]
    .to_dict()
)

reference_names = list(reference.keys())

def match_category(merchant_name, threshold=85):
    if merchant_name in reference:
        return reference[merchant_name], 100
    else:
        best_match, score, _ = process.extractOne(
            merchant_name, reference_names, scorer=fuzz.ratio
        )

        if score >= threshold:
            return reference[best_match], score
        else:
            return None, score

In [55]:
missing_rows = transactions_df[transactions_df["merchant_category"].isna()].copy()

missing_rows[["matched_category", "match_score"]] = missing_rows["merchant_name"].apply(
lambda name: pd.Series(match_category(name))
)

display(missing_rows[["merchant_name", "matched_category", "match_score"]].head(20))

,merchant_name,matched_category,match_score
13002,Transport_01,transport,100
17321,Restaurants_12,restaurants,100
24205,Entertainment_08,entertainment,100
26191,Groceries_23,groceries,100
28612,Entertainment_09,entertainment,100
35484,Transport_08,transport,100
36671,Utilities_24,utilities,100
38884,Travel_15,travel,100
46277,Insurance_33,insurance,100
49113,Travel_24,travel,100
